# Extend trained depth sweeps to depths 11–15

Run the cells in order. Set `PROJECT_ROOT` to the repository directory containing `core` and `tests`. Set `RESULTS_DIR` to the existing results directory from the original notebook. Existing depth 1–10 results are retained. Only load trusted checkpoint and pickle files.

In [1]:
import copy
import math
import os
import pickle
import shutil
import sys
from collections.abc import Mapping
from pathlib import Path

import numpy as np
import optuna
import pennylane as qml
import torch
from torch import nn
from itertools import groupby
from numbers import Integral
from uuid import uuid4

PROJECT_ROOT = Path(
    r"C:\Users\Adrian\PycharmProjects\Effective_dimension_tds"
)
RESULTS_DIR = PROJECT_ROOT / "results"

if not (PROJECT_ROOT / "core").is_dir():
    raise FileNotFoundError(f"Set PROJECT_ROOT to your repository: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.trainers.hyperparameter_trainer import HyperparameterTrainer
from core.model.models import GQNN
from tests.led_depth_sweep import sweep_led_by_depth
from tests.ged_depth_sweep import sweep_ged_by_depth

SEEDS = tuple(range(2026, 2029))
NEW_DEPTHS = tuple(range(16, 22))
N_QUBITS = 4
N_TRIALS = 20

torch.set_default_dtype(torch.float64)

print(f"Project: {PROJECT_ROOT}")
print(f"Existing results: {RESULTS_DIR}")

Project: C:\Users\Adrian\PycharmProjects\Effective_dimension_tds
Existing results: C:\Users\Adrian\PycharmProjects\Effective_dimension_tds\results


## Load datasets and initialize the trainer

All expected result files are checked before training begins. Dataset paths are resolved from the repository root, independently of the notebook location.

In [2]:
dataset_path = PROJECT_ROOT / "core" / "datasets"
chart_name = "Mackey_Glass_tau_30"
training_path = dataset_path / f"train_dataset_{chart_name}.pt"
validating_path = dataset_path / f"val_dataset_{chart_name}.pt"
test_path = dataset_path / f"test_dataset_{chart_name}.pt"

required_paths = [training_path, validating_path, test_path]
required_paths += [
    RESULTS_DIR / f"trained_prefix_led_ged_results_{seed}.pkl"
    for seed in SEEDS
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing files:\n" + "\n".join(missing_paths))

trainer = HyperparameterTrainer(
    training_path=training_path,
    validating_path=validating_path,
    testing_path=test_path,
    criterion=nn.MSELoss(),
)

test_dataset = torch.load(test_path, map_location="cpu", weights_only=False)
inputs = test_dataset.tensors[0].detach().clone().to(
    device="cpu", dtype=torch.get_default_dtype()
)

## Model configuration and training

An interrupted Optuna study is not resumed unless its final model checkpoint was saved.

In [3]:
def make_config(depth, params):
    return {
        "model_config": {
            "n_layers": depth,
            "n_qubits": N_QUBITS,
            "quantum_device": "default.qubit",
            "interface": "torch",
            "diff_method": "best",
            "fm_style": params["fm_style"],
            "meas": [0],
        },
        "training_config": {
            "number_of_training_workers": 0,
            "number_of_validating_workers": 0,
            "number_of_testing_workers": 0,
            "batch_size": 1,
            "device": "cpu",
            "epochs": 20,
            "optimizer": {
                "name": "Adam",
                "lr": params["lr"],
                "momentum": 0.0,
                "weight_decay": params["weight_decay"],
            },
            "regularization": {
                "type": params["regularization"],
                "lambda": params.get("lambda", 0.0),
            },
        },
    }


def sample_params(trial):
    regularization = trial.suggest_categorical(
        "regularization", ["none", "l1", "l2"]
    )
    params = {
        "fm_style": trial.suggest_categorical("fm_style", ["zzfm", "iqp", "X"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-1, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "regularization": regularization,
    }
    if regularization != "none":
        params["lambda"] = trial.suggest_float("lambda", 1e-6, 1e-4, log=True)
    return params


def load_model(record):
    config = record["config"]["model_config"]
    net = GQNN(
        n_layers=config["n_layers"],
        n_qubits=config["n_qubits"],
        quantum_device=qml.device(
            config["quantum_device"], wires=config["n_qubits"]
        ),
        interface=config["interface"],
        diff_method=config["diff_method"],
        fm_style=config["fm_style"],
        meas=config["meas"],
    )
    net.qlayers.load_state_dict(record["state_dict"])
    net.qlayers.eval()
    return net


def train_depth(depth, seed):
    best = {"loss": math.inf, "config": None, "state_dict": None}

    def objective(trial):
        torch.manual_seed(seed + depth)
        config = make_config(depth, sample_params(trial))
        net, metrics = trainer.train_model(trial, config, GQNN)
        loss = float(metrics["validating_loss"][-1])
        if not math.isfinite(loss):
            raise ValueError("Non-finite validation loss")
        if loss < best["loss"]:
            best["loss"] = loss
            best["config"] = copy.deepcopy(config)
            best["state_dict"] = {
                key: value.detach().cpu().clone()
                for key, value in net.qlayers.state_dict().items()
            }
        return loss

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed + depth),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=5, n_warmup_steps=5
        ),
    )
    study.optimize(objective, n_trials=N_TRIALS)
    if best["state_dict"] is None:
        raise RuntimeError(f"No successful model: seed={seed}, depth={depth}")
    print(
        f"Seed {seed}, depth {depth}: validation MSE = {study.best_value:.6f}, "
        f"parameters = {study.best_params}"
    )
    return {"config": best["config"], "state_dict": best["state_dict"]}

## Persistence and result merging

The merger supports dictionaries keyed by integer depths, or dictionaries with a `depths` field and per-depth sequences or arrays along axis 0. Other fields must be identical metadata. Unsupported structures raise an error. Newly computed metrics are saved separately before merging, so they remain available if the merger needs adjustment.

In [4]:
def save_pickle(data, path):
    temporary = path.with_name(path.name + ".tmp")
    try:
        with temporary.open("wb") as file:
            pickle.dump(data, file, protocol=pickle.HIGHEST_PROTOCOL)
            file.flush()
            os.fsync(file.fileno())
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def save_checkpoint(record, path):
    temporary = path.with_name(path.name + ".tmp")
    try:
        torch.save(record, temporary)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def values_equal(left, right):
    if isinstance(left, torch.Tensor) and isinstance(right, torch.Tensor):
        return torch.equal(left, right)
    if isinstance(left, np.ndarray) and isinstance(right, np.ndarray):
        return np.array_equal(left, right, equal_nan=True)
    try:
        return bool(left == right)
    except (TypeError, ValueError, RuntimeError):
        return False


def merge_values(old, new, old_count, new_count, location):
    if isinstance(old, Mapping) and isinstance(new, Mapping):
        if set(old) != set(new):
            raise ValueError(f"Different keys at {location}")
        return {
            key: merge_values(
                old[key], new[key], old_count, new_count, f"{location}.{key}"
            )
            for key in old
        }
    if isinstance(old, np.ndarray) and isinstance(new, np.ndarray):
        if old.ndim and new.ndim:
            if old.shape[0] == old_count and new.shape[0] == new_count:
                return np.concatenate((old, new), axis=0)
    if isinstance(old, torch.Tensor) and isinstance(new, torch.Tensor):
        if old.ndim and new.ndim:
            if old.shape[0] == old_count and new.shape[0] == new_count:
                return torch.cat((old, new), dim=0)
    if isinstance(old, (list, tuple)) and isinstance(new, type(old)):
        if len(old) == old_count and len(new) == new_count:
            return old + new
    if isinstance(old, range) and isinstance(new, range):
        if len(old) == old_count and len(new) == new_count:
            return list(old) + list(new)
    if values_equal(old, new):
        return copy.deepcopy(old)
    raise ValueError(f"Cannot safely merge {location}. Adapt the merger to the sweep output.")


def merge_sweep_results(old, new):
    if hasattr(old, "depths") and hasattr(new, "depths"):
        if type(old) is not type(new):
            raise TypeError("Sweep result types do not match")

        if old.theoretical_dataset_size != new.theoretical_dataset_size:
            raise ValueError("Different theoretical_dataset_size values")

        if set(old.depths) & set(new.depths):
            raise ValueError("Existing and new sweep depths overlap")

        rows = {}

        for result in (old, new):
            if not (
                len(result.depths)
                == len(result.means)
                == len(result.standard_errors)
                == len(set(result.depths))
            ):
                raise ValueError("Misaligned fields or duplicate depths")

            if set(result.samples) != set(result.depths):
                raise ValueError("Sample keys do not match sweep depths")

            for index, depth in enumerate(result.depths):
                rows[depth] = (
                    result.means[index],
                    result.standard_errors[index],
                    result.samples[depth],
                )

        depths = sorted(rows)

        return type(old)(
            depths=depths,
            means=[rows[depth][0] for depth in depths],
            standard_errors=[rows[depth][1] for depth in depths],
            samples={depth: rows[depth][2] for depth in depths},
            theoretical_dataset_size=old.theoretical_dataset_size,
        )

    raise TypeError(
        f"Unsupported sweep result types: "
        f"{type(old).__name__} and {type(new).__name__}"
    )

## Run the extension

LED uses each run seed. The original result file is backed up once before replacement. Run only one instance of this extension notebook at a time.

In [ ]:
requested_depths = tuple(NEW_DEPTHS)

if not requested_depths:
    raise ValueError("NEW_DEPTHS must not be empty")

if any(
    isinstance(depth, bool) or not isinstance(depth, Integral) or depth < 1
    for depth in requested_depths
):
    raise ValueError("NEW_DEPTHS must contain positive integers")

requested_depths = tuple(sorted({int(depth) for depth in requested_depths}))
required_fields = {"led_result", "ged_result", "test_losses"}

for seed in SEEDS:
    results_path = RESULTS_DIR / f"trained_prefix_led_ged_results_{seed}.pkl"

    with results_path.open("rb") as file:
        existing = pickle.load(file)

    if not required_fields.issubset(existing):
        raise ValueError(f"Missing result fields in {results_path}")

    completed = set(existing["test_losses"])

    for key in ("led_result", "ged_result"):
        stored_depths = list(existing[key].depths)
        if len(stored_depths) != len(set(stored_depths)):
            raise ValueError(f"Duplicate depths in {key}: {results_path}")
        if set(stored_depths) != completed:
            raise ValueError(
                f"Inconsistent depths in {key} and test_losses: {results_path}"
            )

    missing_depths = sorted(set(requested_depths) - completed)

    if not missing_depths:
        print(f"Seed {seed}: all requested depths already present, skipping")
        continue

    print(f"Seed {seed}: missing depths {missing_depths}")

    batches = [
        tuple(depth for _, depth in group)
        for _, group in groupby(
            enumerate(missing_depths),
            key=lambda item: item[1] - item[0],
        )
    ]

    for depths in batches:
        first_depth, last_depth = depths[0], depths[-1]
        expected_depths = set(depths)

        extension_path = RESULTS_DIR / (
            f"depth_{first_depth}_{last_depth}_extension_{seed}.pkl"
        )

        if extension_path.exists():
            with extension_path.open("rb") as file:
                extension = pickle.load(file)

            print(
                f"Seed {seed}: loaded saved metrics "
                f"for depths {first_depth}–{last_depth}"
            )
        else:
            records = {}

            for depth in depths:
                checkpoint_path = RESULTS_DIR / f"trained_depth_{depth}_{seed}.pt"

                if checkpoint_path.exists():
                    record = torch.load(
                        checkpoint_path,
                        map_location="cpu",
                        weights_only=False,
                    )
                    print(f"Seed {seed}, depth {depth}: loaded checkpoint")
                else:
                    record = train_depth(depth, seed)

                model_config = record["config"]["model_config"]

                if (
                    model_config["n_layers"] != depth
                    or model_config["n_qubits"] != N_QUBITS
                ):
                    raise ValueError(
                        f"Incompatible model configuration: {checkpoint_path}"
                    )

                if not checkpoint_path.exists():
                    save_checkpoint(record, checkpoint_path)

                records[depth] = record

            def model_factory(depth):
                return load_model(records[depth])

            led_result = sweep_led_by_depth(
                model_factory=model_factory,
                inputs=inputs,
                depths=depths,
                theoretical_dataset_size=1_000,
                n_theta=10,
                epsilon=0.1,
                repetitions=5,
                normalize=True,
                seed=seed,
                min_probability=1e-12,
            )

            ged_result = sweep_ged_by_depth(
                model_factory=model_factory,
                inputs=inputs,
                depths=depths,
                theoretical_dataset_size=1_000,
                n_theta=10,
                normalize=True,
                repetitions=5,
                seed=seed,
                min_probability=1e-12,
            )

            test_losses = {
                depth: trainer.test_model(
                    records[depth]["config"],
                    model_factory(depth),
                )
                for depth in depths
            }

            extension = {
                "led_result": led_result,
                "ged_result": ged_result,
                "test_losses": test_losses,
            }

            save_pickle(extension, extension_path)

        if not required_fields.issubset(extension):
            raise ValueError(f"Missing result fields in {extension_path}")

        if set(extension["test_losses"]) != expected_depths:
            raise ValueError(
                f"Unexpected test-loss depths in {extension_path}: "
                f"expected {list(depths)}, "
                f"found {sorted(extension['test_losses'])}"
            )

        for key in ("led_result", "ged_result"):
            stored_depths = list(extension[key].depths)

            if (
                len(stored_depths) != len(expected_depths)
                or set(stored_depths) != expected_depths
            ):
                raise ValueError(
                    f"Unexpected or duplicate depths in {key}: {extension_path}"
                )

        updated = dict(existing)

        for key in ("led_result", "ged_result"):
            updated[key] = merge_sweep_results(existing[key], extension[key])

        updated["test_losses"] = dict(
            sorted(
                {
                    **existing["test_losses"],
                    **extension["test_losses"],
                }.items()
            )
        )

        expected_merged_depths = set(existing["test_losses"]) | expected_depths

        for key in ("led_result", "ged_result"):
            merged_depths = list(updated[key].depths)

            if (
                len(merged_depths) != len(expected_merged_depths)
                or set(merged_depths) != expected_merged_depths
            ):
                raise ValueError(
                    f"Invalid merged {key} for seed {seed}"
                )

        backup_path = results_path.with_name(
            results_path.name
            + f".before_depth_{first_depth}_{last_depth}.bak"
        )

        if not backup_path.exists():
            shutil.copy2(results_path, backup_path)

        save_pickle(updated, results_path)
        existing = updated

        print(f"Seed {seed}: appended depths {first_depth}–{last_depth}")

[I 2026-10-31 11:29:22,233] A new study created in memory with name: no-name-4abb5e42-f5ee-4cc6-a2b7-f623af7c9928


Seed 2026: missing depths [16, 17, 18, 19, 20, 21]


Trial 0:   0%|          | 0/20 [00:00<?, ?it/s]

## Verify saved depth coverage

This checks test-loss depth coverage. Sweep depth coverage is validated by the merger before saving.

In [ ]:
expected_depths = set(NEW_DEPTHS)

for seed in SEEDS:
    path = RESULTS_DIR / f"trained_prefix_led_ged_results_{seed}.pkl"

    with path.open("rb") as file:
        result = pickle.load(file)

    if not {"led_result", "ged_result", "test_losses"}.issubset(result):
        raise ValueError(f"Missing result fields in {path}")

    completed = set(result["test_losses"])
    missing = sorted(expected_depths - completed)

    if missing:
        raise ValueError(
            f"Incomplete depth coverage for seed {seed}: missing {missing}"
        )

    for key in ("led_result", "ged_result"):
        depths = list(result[key].depths)

        if len(depths) != len(set(depths)) or set(depths) != completed:
            raise ValueError(
                f"Inconsistent or duplicate depths in {key} for seed {seed}"
            )

    print(
        f"Seed {seed}: verified requested depths {sorted(expected_depths)}"
    )